## Lab 17 — Decision Trees: Pruning and Model Selection

### Instructions

Continue the rental-demand problem. Compare tree sizes, prune a grown tree, and search a small parameter grid.
Choose settings with training and validation data before evaluating the reserved final test period.

**Time:** 80 minutes. **Prerequisites:** basic Python, pandas, and train/validation evaluation.
Open this file using **Colab → File → Upload notebook**. Use a CPU runtime.

The notebook uses short code cells and direct `fit`, `predict`, and `score` calls.
This instructor edition contains completed code, expected answers, and reference outputs.
Distribute only the student edition. Restart and run all cells before teaching.

The dataset downloads automatically. If downloading fails, download the UCI ZIP from the link below,
upload `hour.csv` through Colab's Files pane, and rerun the loading cell.


### Industry Problem Statement — Bike Rental Demand

A shared-bike service such as **Lime, whose vehicles can be rented through Uber**, needs to plan for changes in demand.
Too little preparation can leave customers without bikes; unnecessary preparation uses operating resources.

Use decision trees to answer two questions:

1. **Classification:** Will an hour have at least **300 rentals**?
2. **Regression:** How many rentals are expected during that hour?

Use real public **Capital Bikeshare** records from 2011–2012. Uber and Lime provide industry context;
these are not their data or internal models. The 300-rental threshold is a fixed classroom assumption.

The output is an hourly demand report, not a station-level allocation plan. The data contain observed total rentals,
not lost requests. Weather is recorded historical weather; a future-hour service would need prediction-time forecasts.

[Industry context](https://help.li.me/hc/en-us/articles/1260805266589-Renting-Lime-vehicles-with-Uber) ·
[Public dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)


## Step 1: Recreate the Data

This notebook runs on its own. The supplied cells recreate the same six inputs, busy-hour target, rental-count target,
and chronological 60/20/20 split from Lab 16. The final test rows remain unused until Step 7.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.tree import plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

print("scikit-learn version:", sklearn.__version__)


In [ ]:
# Supplied data-loading cell. No extra package is needed.
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
if not Path("hour.csv").exists():
    if not Path("bike_sharing.zip").exists():
        urlretrieve(url, "bike_sharing.zip")
    with ZipFile("bike_sharing.zip") as archive:
        Path("hour.csv").write_bytes(archive.read("hour.csv"))

data = pd.read_csv("hour.csv")
data = data.sort_values(["dteday", "hr"]).reset_index(drop=True)
print("Number of hourly records:", len(data))


In [ ]:
features = ["hr", "temp", "hum", "windspeed", "workingday", "yr"]
data["busy"] = (data["cnt"] >= 300).astype(int)


In [ ]:
# Keep earlier observations before later ones: shuffle=False is essential.
development, test = train_test_split(data, test_size=0.20, shuffle=False)
train, validation = train_test_split(development, test_size=0.25, shuffle=False)

X_train = train[features]
X_val = validation[features]
X_test = test[features]
y_train = train["busy"]
y_val = validation["busy"]
y_test = test["busy"]

print("Train:", len(train), "Validation:", len(validation), "Test:", len(test))


In [ ]:
# Same rows, different target: count rather than busy/not busy.
yr_train = train["cnt"]
yr_val = validation["cnt"]
yr_test = test["cnt"]


assert len(train) + len(validation) + len(test) == len(data)
assert train.index.max() < validation.index.min() < test.index.min()
assert not {"cnt", "casual", "registered", "busy", "instant"}.intersection(features)
print("Checks passed. The final test set remains unused until Lab 17.")


### Entropy Is a Split Criterion

Gini and entropy measure mixed class labels. Information gain measures their reduction after a split;
C4.5's gain ratio adjusts information gain by split information.
The code below changes the criterion, but still uses scikit-learn's CART-style binary implementation.


In [ ]:
# TODO: Create a DecisionTreeClassifier with criterion='entropy', max_depth=2, and random_state=42
# entropy_tree = ...
# Fit the tree on X_train and y_train
# ...
# Print the text representation of the tree
# print(export_text(entropy_tree, feature_names=features))

### Check Your Understanding

Does setting criterion="entropy" turn DecisionTreeClassifier into ID3?

**Instructor answer:** No. An impurity criterion and a complete tree-building algorithm are different. The estimator still uses CART-style binary splits; it does not become ID3 or implement C4.5 gain ratio.


## Step 2: Diagnose the Baseline

Fit an unrestricted tree. Compare training and validation accuracy before changing any settings.
For this industry problem, the important question is whether the rule works on later hours.


### TODO 17.1

Fit the unrestricted classifier and calculate its training and validation accuracy.


In [ ]:
# TODO: Fit the unrestricted classifier
# baseline = ...
# Fit the tree on X_train and y_train
# ...

print("Training accuracy:", baseline.score(X_train, y_train))
print("Validation accuracy:", baseline.score(X_val, y_val))
print("Depth:", baseline.get_depth())
print("Leaves:", baseline.get_n_leaves())

In [ ]:
# TODO: Get the leaf IDs reached by each training row and compute their value counts
# leaf_ids = ...
# leaf_sizes = ...

print(leaf_sizes.describe())

plt.figure(figsize=(7, 3))
plt.hist(leaf_sizes, bins=np.geomspace(0.5, leaf_sizes.max()+1, 25))
plt.xscale("log")
plt.xlabel("Training rows per leaf (log scale)")
plt.ylabel("Number of leaves")
plt.show()

### Check Your Understanding

What suggests possible overfitting? Can the later-period validation gap have another explanation?

**Instructor answer:** Quote the train–validation gap and the number or distribution of small leaves. Those observations motivate controlling complexity. Changes in weather, season, or rental demand between periods can also affect performance, so depth or small leaves alone are not proof of overfitting.


## Step 3: Limit Growth Before Pruning

| Setting | Meaning |
|---|---|
| `max_depth` | Maximum number of edges from root to leaf |
| `min_samples_split` | Minimum parent size required to attempt a split |
| `min_samples_leaf` | Minimum rows in each resulting child |
| `max_leaf_nodes` | Maximum leaves |
| `max_features` | Inputs considered when searching a split |
| `min_impurity_decrease` | Minimum impurity improvement weighted over the full training set |

Vary one setting at a time. `criterion` and `splitter` are not pruning-strength controls;
reducing `max_features` can add randomness and does not guarantee improvement.


### TODO 17.2a

Vary only maximum depth. Fit each new tree and record both accuracy values.


In [ ]:
# Completed solution 17.2a
depths = [1, 2, 3, 4, 6, None]
pre_trees = []
pre_results = []
for depth in depths:
    # TODO: Create a DecisionTreeClassifier with the current max_depth and random_state=42
    # tree = ...
    tree.fit(X_train, y_train)
    pre_trees.append(tree)
    pre_results.append(["depth=" + str(depth), tree.score(X_train, y_train),
                        tree.score(X_val, y_val), tree.get_depth(), tree.get_n_leaves()])

### TODO 17.2b

Now vary only minimum leaf size. Keep maximum depth unrestricted for this separate experiment.


In [ ]:
# Completed solution 17.2b
minimums = [1, 2, 5, 10, 20]
for minimum in minimums:
    # TODO: Create a DecisionTreeClassifier with the current min_samples_leaf and random_state=42
    # tree = ...
    tree.fit(X_train, y_train)
    pre_trees.append(tree)
    pre_results.append(["min_leaf=" + str(minimum), tree.score(X_train, y_train),
                        tree.score(X_val, y_val), tree.get_depth(), tree.get_n_leaves()])

columns = ["setting", "train_accuracy", "val_accuracy", "depth", "leaves"]
pre_table = pd.DataFrame(pre_results, columns=columns)
display(pre_table)

In [ ]:
# Supplied comparison plot for the depth experiment.
depth_table = pre_table.iloc[:6]
plt.figure(figsize=(6, 3))
plt.plot(depth_table["setting"], depth_table["train_accuracy"], "o-", label="Train")
plt.plot(depth_table["setting"], depth_table["val_accuracy"], "o-", label="Validation")
plt.ylabel("Accuracy")
plt.xticks(rotation=20)
plt.legend()
plt.show()


In [ ]:
leaf_table = pre_table.iloc[6:]
plt.figure(figsize=(6, 3))
plt.plot(minimums, leaf_table["train_accuracy"], "o-", label="Train")
plt.plot(minimums, leaf_table["val_accuracy"], "o-", label="Validation")
plt.xlabel("Minimum samples per leaf")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


### Select One Pre-Pruned Candidate

Use highest validation accuracy, then fewer leaves, then smaller actual depth. If all tie, keep experiment order.
`sort_values` sorts the table by that rule; its row index selects the matching tree from the list.


In [ ]:
pre_table["order"] = range(len(pre_table))
ranked_pre = pre_table.sort_values(["val_accuracy", "leaves", "depth", "order"],
                                   ascending=[False, True, True, True])
best_pre_index = ranked_pre.index[0]
pre_tree = pre_trees[best_pre_index]
display(ranked_pre.head(1))


### Check Your Understanding

A six-row parent proposes a 1/5 split. What do min_samples_split=4 and min_samples_leaf=2 allow?

**Instructor answer:** The parent meets min_samples_split=4, so a split may be attempted. The proposed split violates min_samples_leaf=2 because one child would have only one row. Minimum leaf size sets a support requirement, not a guarantee of reliability.


## Step 4: Prune a Grown Tree

Cost-complexity pruning balances training impurity against the number of leaves:

$$R_\alpha(T)=R(T)+\alpha L(T)$$

$R(T)$ is total sample-weighted leaf impurity in scikit-learn. $L(T)$ is leaf count and $\alpha\geq0$.
This is not a validation-accuracy formula. Generate candidate alphas from training data only;
choose among them using validation accuracy. Keep the root-only endpoint to see excessive pruning.


### TODO 17.3

Calculate the pruning path and fit a fresh tree for every alpha. The comparison code is supplied.


In [ ]:
# Completed solution 17.3
path = baseline.cost_complexity_pruning_path(X_train, y_train)
alphas = path.ccp_alphas
pruned_trees = []
pruning_results = []
for alpha in alphas:
    # TODO: Create a DecisionTreeClassifier with the current ccp_alpha and random_state=42, and fit it on X_train and y_train
    # tree = ...
    # Fit the tree
    # ...
    pruned_trees.append(tree)
    pruning_results.append([alpha, tree.score(X_train, y_train), tree.score(X_val, y_val),
                            tree.get_depth(), tree.get_n_leaves()])

In [ ]:
columns = ["alpha", "train_accuracy", "val_accuracy", "depth", "leaves"]
pruning_table = pd.DataFrame(pruning_results, columns=columns)
pruning_table["order"] = range(len(pruning_table))
ranked_pruning = pruning_table.sort_values(["val_accuracy", "leaves", "depth", "order"],
                                          ascending=[False, True, True, True])
best_pruning_index = ranked_pruning.index[0]
pruned_tree = pruned_trees[best_pruning_index]
display(ranked_pruning.head(5))
assert pruning_table.iloc[-1]["leaves"] == 1


### Supplied Pruning Plots

The alpha scale is linear near zero and logarithmic beyond it, so both small penalties and the root-only endpoint are visible.


In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(alphas, path.impurities)
plt.xscale("symlog", linthresh=0.0001)
plt.xlim(left=0)
plt.xlabel("Alpha")
plt.ylabel("Total leaf impurity")
plt.show()


In [ ]:
plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1)
plt.plot(alphas, pruning_table["leaves"])
plt.xscale("symlog", linthresh=0.0001)
plt.xlim(left=0)
plt.xlabel("Alpha")
plt.ylabel("Leaves")
plt.subplot(1, 2, 2)
plt.plot(alphas, pruning_table["depth"])
plt.xscale("symlog", linthresh=0.0001)
plt.xlim(left=0)
plt.xlabel("Alpha")
plt.ylabel("Depth")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(alphas, pruning_table["train_accuracy"], label="Train")
plt.plot(alphas, pruning_table["val_accuracy"], label="Validation")
plt.xscale("symlog", linthresh=0.0001)
plt.xlim(left=0)
plt.xlabel("Alpha")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


### Check Your Understanding

Must pruning improve validation accuracy? What does a root-only classifier predict?

**Instructor answer:** No: pruning can remove useful patterns. A root-only tree predicts the training majority class for every hour. Validation accuracy has no monotonic guarantee as alpha increases. Prefer fewer leaves only according to the declared tie rule, not as proof of future superiority.


### Reduced-Error Pruning: A Short Decision Exercise

These are hypothetical predictions on the same 20 validation cases, separate from the rental dataset.
In a full reduced-error procedure, replace a subtree with its training-majority class leaf.
Accept if validation accuracy does not decrease; process children before parents and update the baseline after acceptance.
This exercise implements only that decision rule, not a complete traversal or a scikit-learn setting.


In [ ]:
true_labels = np.array([0] * 10 + [1] * 10)
current = true_labels.copy()
current[[1, 5, 12]] = 1 - current[[1, 5, 12]]
trial_A = true_labels.copy()
trial_A[[2, 5, 12]] = 1 - trial_A[[2, 5, 12]]
trial_B = true_labels.copy()
trial_B[[2, 5, 12, 18]] = 1 - trial_B[[2, 5, 12, 18]]


### TODO 17.4

Complete the accuracy calculation and acceptance rule. Trial B is a proposal after A has been accepted.


In [ ]:
# Completed solution 17.4
current_accuracy = accuracy_score(true_labels, current)
for name, proposal in [("A", trial_A), ("B after A", trial_B)]:
    # TODO: Calculate the proposed model's accuracy and decide whether to accept it (accuracy >= current_accuracy)
    # new_accuracy = ...
    # accept = ...
    print(name, "current:", current_accuracy, "proposed:", new_accuracy, "accept:", accept)
    if accept:
        current = proposal.copy()
        current_accuracy = new_accuracy

assert np.array_equal(current, trial_A)

### Check Your Understanding

Why retain the version with A pruned but not B?

**Instructor answer:** A retains 17/20 correct predictions (85%) with a simpler model, so it is accepted. B would reduce accuracy to 16/20 (80%) relative to the updated baseline, so it is rejected.


## Step 5: Search Parameter Combinations

`GridSearchCV` compares combinations. Use three forward folds inside the training period:
each fold fits on earlier recorded hours and scores on later ones. It does not use the reserved final test period.
Because some hours are missing, fold calendar durations need not be identical; folds contain equal numbers of recorded validation rows.

The grid contains 27 combinations: **81 fold fits plus one refit**. Keep the alpha grid fixed and separate from
the earlier validation-selected path. The six inputs are numeric, so no learned preprocessing is needed here.
If preprocessing is added later, include it inside the searched Pipeline.


In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

param_grid = {
    "max_depth": [2, 4, None],
    "min_samples_leaf": [1, 5, 10],
    "ccp_alpha": [0.0, 0.005, 0.02]
}
cv = TimeSeriesSplit(n_splits=3)


### TODO 17.5

Complete the metric and fit the search on training data only.


In [ ]:
# Completed solution 17.5
# TODO: Create a GridSearchCV with DecisionTreeClassifier(random_state=42), param_grid, cv=cv, and scoring='accuracy'
# grid = ...
# Fit the grid search on X_train and y_train
# ...
grid_tree = grid.best_estimator_

print("Best parameters:", grid.best_params_)
print("Best mean CV score:", grid.best_score_)

In [ ]:
cv_results = pd.DataFrame(grid.cv_results_)
display(cv_results.sort_values("rank_test_score")[[
    "params", "mean_train_score", "mean_test_score", "std_test_score"
]].head(5))
assert len(cv_results) == 27 and grid.n_splits_ == 3


### Check Your Understanding

What does mean_test_score mean here? Does the best grid score guarantee the globally best tree?

**Instructor answer:** It is the average score on held-out cross-validation folds within the training period, not the reserved final test set. The search only compares the specified candidates and folds. A training–fold gap can suggest overfitting, while fold variability can reflect seasonal changes or sampling differences. Exact CV ties use GridSearchCV candidate order.


## Step 6: Apply the Same Idea to Regression

Use negative MSE because scikit-learn maximizes search scores. Negate `best_score_` to report positive MSE.
The six candidates require 18 fold fits plus a refit. Regression also supports pruning, but its impurity scale
differs from classification, so classification alpha values should not be copied blindly.


### TODO 17.6

Complete the scoring metric, fit call, and conversion back to positive MSE.


In [ ]:
# Completed solution 17.6
# TODO: Create a GridSearchCV with DecisionTreeRegressor(random_state=42), the parameter grid, cv=cv, and scoring='neg_mean_squared_error'
# reg_grid = ...
# Fit the grid search on X_train and yr_train
# ...
reg_grid_tree = reg_grid.best_estimator_

print("Best regression parameters:", reg_grid.best_params_)
print("Best CV MSE:", -reg_grid.best_score_)

In [ ]:
reg_baseline = DecisionTreeRegressor(random_state=42)
reg_baseline.fit(X_train, yr_train)
reg_models = [reg_baseline, reg_grid_tree]
reg_names = ["Unrestricted", "Grid winner"]
reg_results = []
for name, model in zip(reg_names, reg_models):
    reg_results.append([name, mean_squared_error(yr_train, model.predict(X_train)),
                        mean_squared_error(yr_val, model.predict(X_val)), model.get_depth(), model.get_n_leaves()])

reg_table = pd.DataFrame(reg_results, columns=["model", "train_MSE", "val_MSE", "depth", "leaves"])
display(reg_table)


In [ ]:
profile = pd.DataFrame([X_train.iloc[0].to_dict()] * 24)
profile["hr"] = range(24)
plt.figure(figsize=(8, 3))
plt.step(profile["hr"], reg_baseline.predict(profile), where="mid", label="Unrestricted")
plt.step(profile["hr"], reg_grid_tree.predict(profile), where="mid", label="Grid winner")
plt.xlabel("Hour; other inputs fixed to first training row")
plt.ylabel("Predicted rentals")
plt.legend()
plt.show()
if reg_baseline.get_params() == reg_grid_tree.get_params():
    print("The search selected the baseline settings; the curves overlap.")


### Check Your Understanding

Why report positive MSE? What would RMSE communicate to someone planning rental operations?

**Instructor answer:** The negative sign is a maximization convention; MSE itself is nonnegative. RMSE is the square root of MSE and has rental-count units per hourly record. It is easier to interpret than squared units, but it is not a direct measure of fleet actions or financial benefit.


## Step 7: Choose Settings Before Testing

Compare the four classification candidates in the order below. Choose highest validation accuracy, then fewer leaves,
smaller depth, then original order. For regression, choose lowest validation MSE with the same complexity tie rules.
Use full-precision scores when ranking. The sorting code is supplied so the exercise remains focused on model selection.


In [ ]:
models = [baseline, pre_tree, pruned_tree, grid_tree]
names = ["Unrestricted", "Pre-pruned", "Post-pruned", "Grid winner"]
comparison = []
for name, model in zip(names, models):
    comparison.append([name, model.score(X_train, y_train), model.score(X_val, y_val),
                       model.get_depth(), model.get_n_leaves()])
comparison = pd.DataFrame(comparison, columns=["model", "train_accuracy", "val_accuracy", "depth", "leaves"])
comparison["order"] = range(len(comparison))
display(comparison)


In [ ]:
ranked = comparison.sort_values(["val_accuracy", "leaves", "depth", "order"],
                                ascending=[False, True, True, True])
chosen_tree = models[ranked.index[0]]
reg_table["order"] = range(len(reg_table))
ranked_reg = reg_table.sort_values(["val_MSE", "leaves", "depth", "order"],
                                   ascending=[True, True, True, True])
chosen_regressor = reg_models[ranked_reg.index[0]]

print("Classification choice:", ranked.iloc[0]["model"], chosen_tree.get_params())
print("Regression choice:", ranked_reg.iloc[0]["model"], chosen_regressor.get_params())


### Refit on Training Plus Validation

Settings are now fixed. `clone()` creates an unfitted copy with those settings.
Fit it on the whole development period before opening test results. The tree may change after refitting;
the old validation score describes the old fit and is not independent evaluation of the new one.


### TODO 17.7

Refit the selected classifier and regressor on the full development period.


In [ ]:
# Completed solution 17.7
from sklearn.base import clone

final_tree = clone(chosen_tree)
final_tree.fit(development[features], development["busy"])
final_regressor = clone(chosen_regressor)
final_regressor.fit(development[features], development["cnt"])


### TODO 17.8

Predict the test rows once and compute the two final metrics. Do not change settings after seeing these results.


In [ ]:
# Completed solution 17.8
# TODO: Predict the target values on X_test using final_tree and final_regressor
# test_prediction = ...
# reg_test_prediction = ...

test_accuracy = accuracy_score(y_test, test_prediction)
test_mse = mean_squared_error(yr_test, reg_test_prediction)
print("Final test accuracy:", test_accuracy)
print("Final test MSE:", test_mse)
print("Final test RMSE:", np.sqrt(test_mse), "rentals")

### Interpret Errors in the Industry Problem

Accuracy was the declared selection metric. The same final predictions can also show how many busy hours were missed.
If missed busy hours deserve a different metric or cost rule, specify that before a new development cycle with fresh final evaluation data.


In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

print("Busy-hour precision:", precision_score(y_test, test_prediction, zero_division=0))
print("Busy-hour recall:", recall_score(y_test, test_prediction, zero_division=0))
cm = confusion_matrix(y_test, test_prediction, labels=[0, 1])
print("Missed busy hours:", cm[1, 0])
print("Unnecessary busy-hour alerts:", cm[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["Not busy", "Busy"]).plot(cmap="Blues", colorbar=False)
plt.show()


### Predict a New Hour

Use a working-day evening example. Weather values use the source's normalized units.
This hypothetical row has no known outcome; it demonstrates input and prediction, not another accuracy test.


In [ ]:
new_hour = pd.DataFrame([[18, 0.60, 0.55, 0.15, 1, 1]], columns=features)
print("Busy-hour prediction:", final_tree.predict(new_hour))
print("Class probabilities [not busy, busy]:", final_tree.predict_proba(new_hour))
print("Predicted rentals:", final_regressor.predict(new_hour))

plt.figure(figsize=(13, 5))
plot_tree(final_tree, max_depth=3, feature_names=features,
          class_names=["Not busy", "Busy"], filled=True, fontsize=8)
plt.title("Final classifier — display limited to depth 3")
plt.show()
print("Full fitted depth:", final_tree.get_depth())


### Supplied Code: Trace the Full Prediction

The plot may omit deeper nodes. The short loop below prints the actual conditions followed by the new hour.
`decision_path()` identifies visited nodes; `tree_` stores their feature and threshold. Read the output rather than rewriting this code.


In [ ]:
visited_nodes = final_tree.decision_path(new_hour).indices
for node in visited_nodes[:-1]:
    feature = features[final_tree.tree_.feature[node]]
    threshold = final_tree.tree_.threshold[node]
    value = new_hour[feature].iloc[0]
    print(feature, "=", value, "<=", round(threshold, 4), "?", value <= threshold)
print("Final leaf:", final_tree.apply(new_hour)[0])


### Check Your Understanding

What does the final probability mean? Should disappointing test results lead to immediate retuning? Does a good score prove a fleet action will help?

**Instructor answer:** It is the class proportion in the reached leaf after the development refit, not certainty. Report the fixed model’s test results honestly. Retuning based on them requires fresh final evaluation data. System-wide recorded rentals, historical weather, and predictive associations do not establish the benefit of an intervention or station-level allocation.
